# Workshop 5 — From Chatbot to Agent: Tool Calling with NVIDIA NIM

The fifth workshop in the series. Same hosted NIM endpoint, but instead of just asking the model for an answer, we hand it a list of Python functions it's allowed to call. The model picks which function to invoke; our code runs it and feeds the result back. Two tools — a clock and the retriever from Workshop 2.

**One change vs Workshops 1-4:** the chat model switches from `meta/llama-3.1-8b-instruct` to NVIDIA's own `nvidia/llama-3.3-nemotron-super-49b-v1.5`, tuned for reasoning and tool use. Tool calling is noticeably more reliable on it (it's a bigger model, so each call is slower — a fair trade once correctness matters). Same hosted endpoint, only the model string changes.

## Step 1 — Setup (self-contained)

Bundles everything from Workshops 1, 2, and 3 so this notebook stands on its own. Paste your NVIDIA API key when prompted.

In [ ]:
%pip install -q openai numpy

import os, getpass, json
from datetime import datetime
from zoneinfo import ZoneInfo
from openai import OpenAI
import numpy as np

if not os.getenv('NVIDIA_API_KEY'):
    os.environ['NVIDIA_API_KEY'] = getpass.getpass('Paste your NVIDIA API key (starts with nvapi-): ')

client = OpenAI(
    base_url='https://integrate.api.nvidia.com/v1',
    api_key=os.environ['NVIDIA_API_KEY'],
)

MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1.5'   # switched from 3.1-8b: NVIDIA-tuned for reliable tool calling
EMBED_MODEL = 'nvidia/nv-embedqa-e5-v5'

knowledge_base = [
    {'title': 'USC AI Club meeting',
     'text': 'The USC AI Club meets every Thursday at 5 PM in the engineering building, room 204.'},
    {'title': 'USC GPU lab hours',
     'text': 'The USC GPU computing lab is open Monday to Friday from 10 AM to 6 PM.'},
    {'title': 'NVIDIA Developer Program',
     'text': 'USC students can join the NVIDIA Developer Program for free.'},
    {'title': 'Next USC workshop',
     'text': 'The next USC AI Club workshop will cover Retrieval Augmented Generation (RAG).'},
    {'title': 'USC AI/ML office hours',
     'text': 'Office hours for the USC AI/ML faculty are Tuesdays 2-4 PM.'},
    {'title': 'USC robotics lab',
     'text': 'The USC robotics lab requires safety training before students can use the soldering station.'},
    {'title': 'USC tutoring',
     'text': 'Peer tutoring for introductory Python at USC is available Wednesdays from 1 PM to 3 PM.'},
]

def embed_texts(texts, input_type='passage'):
    response = client.embeddings.create(model=EMBED_MODEL, input=texts, extra_body={'input_type': input_type})
    return [np.array(item.embedding, dtype=np.float32) for item in response.data]

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

def retrieve_context(question, k=3):
    q_emb = embed_texts([question], input_type='query')[0]
    scored = [(cosine_similarity(q_emb, item['embedding']), item) for item in knowledge_base]
    scored.sort(key=lambda p: p[0], reverse=True)
    return '\n'.join(f"- {item['text']}" for _, item in scored[:k])

for item, emb in zip(knowledge_base, embed_texts([i['text'] for i in knowledge_base], 'passage')):
    item['embedding'] = emb
print(f'Ready. Embedded {len(knowledge_base)} chunks. Model: {MODEL}')

## Step 2 — Define two tiny tools (plain Python)

In [ ]:
def get_current_time(timezone='America/Los_Angeles'):
    try:
        zone = ZoneInfo(timezone)
    except Exception:
        zone = ZoneInfo('UTC')
    return datetime.now(zone).strftime('%A, %B %d, %Y at %I:%M %p %Z')

def search_campus_info(query):
    return retrieve_context(query, k=3)

# Quick smoke test
print(get_current_time('America/Los_Angeles'))
print(search_campus_info('AI Club'))

## Step 3 — Describe the tools to the model

In [ ]:
tools = [
    {'type': 'function', 'function': {
        'name': 'get_current_time',
        'description': 'Get the current time in an IANA time zone.',
        'parameters': {
            'type': 'object',
            'properties': {'timezone': {'type': 'string', 'description': 'IANA time zone, e.g. America/Los_Angeles or UTC.'}},
        },
    }},
    {'type': 'function', 'function': {
        'name': 'search_campus_info',
        'description': 'Search the USC campus assistant knowledge base for information about USC clubs (including AI Club), labs (GPU lab, robotics lab), workshops, faculty office hours, peer tutoring, and the NVIDIA Developer Program at USC. Always call this for any USC-related question — do not answer from your own knowledge.',
        'parameters': {
            'type': 'object',
            'properties': {'query': {'type': 'string', 'description': 'The USC campus question or search phrase.'}},
            'required': ['query'],
        },
    }},
]

available_tools = {
    'get_current_time': get_current_time,
    'search_campus_info': search_campus_info,
}

## Step 4 — The agent loop

In [ ]:
def ask_agent(question):
    messages = [
        {'role': 'system', 'content': (
            '/no_think\n\nYou are a USC campus assistant with two tools: '
            'get_current_time and search_campus_info. '
            'When the user asks something a tool can answer, call the tool, '
            "then write the final answer based on the tool's result. "
            'Do not call the same tool twice for the same question. '
            'If after using the tools you still cannot find the answer, '
            "reply exactly: I don't have that information — check with the USC AI Club."
        )},
        {'role': 'user', 'content': question},
    ]
    for _ in range(3):                                # hard cap on tool calls
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice='auto', temperature=0.2, max_tokens=400,
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))
        if not message.tool_calls:
            return message.content or 'I could not generate an answer. Please try again.'
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments or '{}')
            result = available_tools[name](**arguments) if name in available_tools else f'Tool {name} is not available.'
            messages.append({'role': 'tool', 'tool_call_id': tool_call.id, 'name': name, 'content': str(result)})
    return 'I hit the tool loop limit.'

## Step 5 — Run it

In [ ]:
for question in [
    'What time is it in Los Angeles?',            # → calls get_current_time
    'When does the USC AI Club meet?',            # → calls search_campus_info
    'Can I get the wifi password?',               # → searches, finds nothing, refuses
]:
    print(f'Q: {question}')
    print(f'A: {ask_agent(question)}\n')

## What just happened

- The clock question makes the model call `get_current_time` and return the real time.
- The AI Club question makes it call `search_campus_info`, read the retrieved chunks, and answer from them.
- The wifi question makes it call `search_campus_info`, see that none of the chunks mention passwords, and fall back to the refusal line.

## The full series

- **Workshop 1:** First API call (the chat function).
- **Workshop 2:** Embedding-based RAG (the retriever this agent reuses).
- **Workshop 3:** Guardrails (the refusal pattern this agent still uses).
- **Workshop 4:** Run NIM on your own GPU (same code, different endpoint).
- **Workshop 5 (this notebook):** Tool calling — chatbot becomes agent.

Star the repo and fork it for your school: https://github.com/torkian/nvidia-nim-workshop